# Evaluasi Sistem ALPR — Branch `zulfan-detector`

Notebook ini mengukur kinerja sistem dari ujung ke ujung, **tinggal jalan** (Run All):

| Bagian | Yang diukur | Keluaran |
|---|---|---|
| **1. Deteksi YOLO** | Precision, Recall, F1, mAP pada data test | Confusion matrix, kurva F1, kurva PR |
| **2. Training YOLO** *(opsional, default mati)* | Proses belajar model | `results.png` (loss & mAP per epoch) |
| **3. Tahap detector** (deskew, warna, polaritas) | Sumber keputusan deskew, kesepakatan proyeksi vs Hough, sebaran warna plat | Grafik + contoh crop |
| **4. OCR nomor & pajak** — versi lama vs versi branch ini | Exact match, CER, Precision/Recall/F1 karakter, angka pajak terbaca | Tabel, confusion matrix karakter |

**Cara pakai:** pilih kernel **`venv (Python 3.12.6)`** milik folder proyek ini, lalu klik **Run All**.
Semua hasil (gambar, CSV) otomatis tersimpan di `Output/evaluasi/<tanggal-jam>/`.

Perkiraan waktu di CPU: Bagian 1 ± 1–3 menit, Bagian 4 ± 5–10 menit untuk 40 plat.

In [ ]:
# ===================== PENGATURAN =====================
SPLIT          = "test"   # data yang dievaluasi: "test" atau "val"
N_PLAT_OCR     = 40       # jumlah plat untuk evaluasi OCR (Bagian 4). Makin banyak makin lama
RUN_YOLO_VAL   = True     # Bagian 1: validasi YOLO
RUN_TRAINING   = False    # Bagian 2: training ulang YOLO (sangat lama di CPU, lebih baik di Colab)
TRAIN_EPOCHS   = 30
IMGSZ          = 640      # ukuran input YOLO, samakan dengan saat model dilatih
BASELINE_REF   = "f4e12af"  # commit main tempat branch ini bercabang, dipakai sebagai "versi lama" di Bagian 4
# ======================================================

import os, sys, re, glob, json, time, types, subprocess, warnings
from datetime import datetime
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

warnings.filterwarnings("ignore")
BASE = os.path.abspath(os.getcwd())
if not os.path.isdir(os.path.join(BASE, "Script")):
    raise RuntimeError("Jalankan notebook ini dari folder proyek (yang berisi folder Script/).")
sys.path.insert(0, BASE)

DATA_DIR = os.path.join(BASE, "Dataset", "Raw", "Indonesian License Plate Dataset")
MODEL_PATH = os.path.join(BASE, "models", "best.pt")
OUT = os.path.join(BASE, "Output", "evaluasi", datetime.now().strftime("%Y%m%d-%H%M%S"))
os.makedirs(OUT, exist_ok=True)

for p, nama in [(DATA_DIR, "dataset"), (MODEL_PATH, "models/best.pt")]:
    print(("OK      " if os.path.exists(p) else "TIDAK ADA ") + nama + " -> " + p)

try:
    import torch
    DEVICE = 0 if torch.cuda.is_available() else "cpu"
except Exception:
    DEVICE = "cpu"
branch = subprocess.run(["git", "branch", "--show-current"], cwd=BASE, capture_output=True, text=True).stdout.strip()
print(f"\nBranch aktif : {branch}\nDevice       : {'GPU' if DEVICE == 0 else 'CPU'}\nFolder hasil : {OUT}")

## Bagian 1 — Evaluasi Deteksi Plat (YOLO)

YOLO di sistem ini hanya punya **satu kelas: plat nomor**. Jadi yang diukur adalah seberapa tepat model **menemukan lokasi** plat:

- **Precision** — dari semua kotak yang ditebak YOLO, berapa persen yang benar-benar plat.
- **Recall** — dari semua plat yang ada di foto, berapa persen yang berhasil ditemukan.
- **F1** — rata-rata harmonis precision dan recall: $F1 = \frac{2PR}{P+R}$.
- **mAP@0.5** — rata-rata presisi kalau kotak dianggap benar saat IoU ≥ 0.5. **mAP@0.5:0.95** lebih ketat (IoU 0.5 sampai 0.95).
- **Confusion matrix** berukuran 2×2: *license_plate* vs *background*. Sel "background → plate" = kotak palsu (FP), sel "plate → background" = plat yang terlewat (FN).

In [ ]:
from ultralytics import YOLO

yolo_summary = {}
if RUN_YOLO_VAL:
    yaml_path = os.path.join(OUT, "data_eval.yaml")
    with open(yaml_path, "w", encoding="utf-8") as f:
        f.write(f"path: \"{DATA_DIR.replace(os.sep, '/')}\"\ntrain: images/train\nval: images/val\ntest: images/test\n"
                "names:\n  0: license_plate\n")

    model = YOLO(MODEL_PATH)
    t0 = time.time()
    res = model.val(data=yaml_path, split=SPLIT, imgsz=IMGSZ, batch=8, device=DEVICE,
                    project=OUT, name="1_yolo_val", exist_ok=True, plots=True, verbose=False)
    P, R = float(res.box.mp), float(res.box.mr)
    yolo_summary = {
        "Precision": P, "Recall": R, "F1": 2 * P * R / (P + R) if P + R else 0.0,
        "mAP@0.5": float(res.box.map50), "mAP@0.5:0.95": float(res.box.map),
    }
    print(f"Selesai dalam {time.time() - t0:.0f} detik\n")
    display(pd.DataFrame([{k: f"{v:.2%}" for k, v in yolo_summary.items()}], index=[f"YOLO ({SPLIT})"]))

    save_dir = str(res.save_dir)
    for pola, judul in [("confusion_matrix.png", "Confusion matrix (jumlah)"),
                        ("confusion_matrix_normalized.png", "Confusion matrix (dinormalisasi)"),
                        ("*F1_curve.png", "Kurva F1 terhadap confidence threshold"),
                        ("*PR_curve.png", "Kurva Precision-Recall")]:
        hits = sorted(glob.glob(os.path.join(save_dir, pola)))
        if hits:
            display(Markdown(f"**{judul}**"))
            display(Image(filename=hits[0], width=560))
else:
    print("Bagian 1 dilewati (RUN_YOLO_VAL = False)")

## Bagian 2 — Training YOLO *(opsional)*

Defaultnya **mati**, karena training di CPU bisa makan berjam-jam. Kalau `RUN_TRAINING = True`, model `yolov8n.pt` dilatih ulang dengan dataset ini dan grafik `results.png` (loss serta mAP tiap epoch) ditampilkan.

Kalau training dimatikan, sel ini mencari hasil training yang sudah ada (misalnya folder `runs/` hasil dari Colab yang disalin ke sini) dan menampilkannya.

In [ ]:
def tampilkan_hasil_training(folder):
    for nama, judul in [("results.png", "Loss & metrik per epoch"),
                        ("confusion_matrix.png", "Confusion matrix (data val)"),
                        ("BoxF1_curve.png", "Kurva F1"), ("F1_curve.png", "Kurva F1")]:
        p = os.path.join(folder, nama)
        if os.path.exists(p):
            display(Markdown(f"**{judul}** — `{os.path.relpath(p, BASE)}`"))
            display(Image(filename=p, width=640))

if RUN_TRAINING:
    yaml_path = os.path.join(OUT, "data_eval.yaml")
    if not os.path.exists(yaml_path):
        with open(yaml_path, "w", encoding="utf-8") as f:
            f.write(f"path: \"{DATA_DIR.replace(os.sep, '/')}\"\ntrain: images/train\nval: images/val\ntest: images/test\n"
                    "names:\n  0: license_plate\n")
    m = YOLO("yolov8n.pt")
    hasil = m.train(data=yaml_path, epochs=TRAIN_EPOCHS, imgsz=IMGSZ, batch=8, device=DEVICE,
                    project=OUT, name="2_training", exist_ok=True)
    tampilkan_hasil_training(str(hasil.save_dir) if hasil is not None else os.path.join(OUT, "2_training"))
    print("Bobot terbaik:", os.path.join(OUT, "2_training", "weights", "best.pt"),
          "\n(salin ke models/best.pt kalau mau dipakai aplikasi)")
else:
    lama = sorted(glob.glob(os.path.join(BASE, "**", "results.png"), recursive=True))
    lama = [p for p in lama if os.sep + "venv" + os.sep not in p]
    if lama:
        print(f"Ditemukan {len(lama)} hasil training lama, menampilkan yang terbaru:")
        tampilkan_hasil_training(os.path.dirname(max(lama, key=os.path.getmtime)))
    else:
        print("Training dimatikan dan belum ada hasil training (results.png) di folder proyek.\n"
              "Minta folder runs/detect/train dari anggota yang melatih model, salin ke folder proyek, lalu jalankan ulang sel ini.")

## Bagian 3 — Tahap Detector: Deskew, Warna Plat, Polaritas

Bagian ini memakai **kotak dari label dataset** supaya yang dinilai murni tahap 2–3 (tidak tercampur kesalahan YOLO).

- **Sumber keputusan deskew** — berapa plat yang dianggap lurus, dikoreksi dengan integral proyeksi, dikonfirmasi Hough, atau memakai Hough sebagai cadangan.
- **Kesepakatan proyeksi vs Hough** — untuk plat yang dikoreksi, seberapa sering dua metode yang berbeda menghasilkan sudut yang sama (selisih ≤ 2.5°). Makin tinggi, makin bisa dipercaya.
- **Sebaran warna plat** — dataset tidak punya label warna, jadi yang bisa ditampilkan adalah sebaran hasil + contoh tiap warna untuk dicek mata.

In [ ]:
from Script.detector import PlateDetector, estimate_skew, estimate_skew_hough, MIN_SKEW_DEG, MIN_SKEW_GAIN
from Script.detector import _crops_from_labels

det = PlateDetector(model_path=MODEL_PATH)
rows3, contoh = [], {}
for ip in sorted(glob.glob(os.path.join(DATA_DIR, "images", SPLIT, "*.jpg"))):
    stem = os.path.splitext(os.path.basename(ip))[0]
    lab = os.path.join(DATA_DIR, "labelswithLP", SPLIT, stem + ".txt")
    if not os.path.exists(lab):
        continue
    for k, (box, crop, gt) in enumerate(_crops_from_labels(ip, lab)):
        r = det.process_crop(crop)
        a_p, gain = estimate_skew(crop)
        a_h, sup = estimate_skew_hough(crop)
        s = r["deskew_status"]
        sumber = ("Lurus" if s.startswith("Lurus") else
                  "Tidak dikoreksi (ragu)" if s.startswith("Tidak dikoreksi") else
                  "Proyeksi & Hough sepakat" if "sepakat" in s else
                  "Hough (cadangan)" if "Hough," in s else "Integral proyeksi saja")
        rows3.append({"plat": f"{stem}_{k}", "gt": gt.replace(" ", ""), "warna": r["plate_color"],
                      "invert": r["is_inverted"], "sudut": r["skew_angle"], "sumber_deskew": sumber,
                      "sudut_proyeksi": a_p, "sudut_hough": a_h})
        contoh.setdefault(r["plate_color"], [])
        if len(contoh[r["plate_color"]]) < 6:
            contoh[r["plate_color"]].append(cv2.cvtColor(cv2.resize(crop, (180, 60)), cv2.COLOR_BGR2RGB))

df3 = pd.DataFrame(rows3)
df3.to_csv(os.path.join(OUT, "3_detector_per_plat.csv"), index=False)
dikoreksi = df3[df3["sudut"].abs() >= MIN_SKEW_DEG]
ada_h = dikoreksi[dikoreksi["sudut_hough"].notna()]
sepakat = (ada_h["sudut_proyeksi"] - ada_h["sudut_hough"]).abs() <= 2.5
print(f"Jumlah plat: {len(df3)} | dikoreksi: {len(dikoreksi)} | "
      f"proyeksi & Hough sepakat: {int(sepakat.sum())}/{len(ada_h)} ({(sepakat.mean() if len(ada_h) else 0):.0%})")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
df3["sumber_deskew"].value_counts().plot.barh(ax=ax[0], color="#4C72B0")
ax[0].set_title("Sumber keputusan deskew"); ax[0].set_xlabel("jumlah plat")
warna_map = {"Hitam": "#333333", "Putih": "#BBBBBB", "Kuning": "#E8B400", "Merah": "#C0392B"}
vc = df3["warna"].value_counts()
vc.plot.bar(ax=ax[1], color=[warna_map.get(w, "#888") for w in vc.index])
ax[1].set_title("Sebaran warna plat (hasil klasifikasi HSV)"); ax[1].set_ylabel("jumlah plat")
for i, v in enumerate(vc.values):
    ax[1].text(i, v, str(v), ha="center", va="bottom")
plt.tight_layout(); plt.savefig(os.path.join(OUT, "3_deskew_dan_warna.png"), dpi=120); plt.show()

fig, axs = plt.subplots(len(contoh), 6, figsize=(13, 1.3 * len(contoh)))
axs = np.atleast_2d(axs)
for i, (w, ims) in enumerate(sorted(contoh.items())):
    for j in range(6):
        axs[i, j].axis("off")
        if j < len(ims):
            axs[i, j].imshow(ims[j])
    axs[i, 0].set_title(w, loc="left", fontsize=10)
plt.suptitle("Contoh crop per warna hasil klasifikasi (cek dengan mata)", y=1.02)
plt.tight_layout(); plt.savefig(os.path.join(OUT, "3_contoh_warna.png"), dpi=110); plt.show()

## Bagian 4 — OCR Nomor Plat & Angka Pajak: versi lama vs versi branch ini

Alurnya **sama persis dengan aplikasi**: foto → YOLO → `detector.py` → `plate_tax_pipeline.analyze_single_plate`.
Kotak hasil YOLO dicocokkan dengan label dataset (IoU ≥ 0.5) untuk mengambil nomor plat yang benar (ground truth).

- **Versi lama** = `plate_tax_pipeline.py` di commit `BASELINE_REF` (titik cabang branch ini dari `main`).
- **Versi baru** = file di branch ini (buang sisa bingkai, cari baris pajak dengan proyeksi, tanpa deskew ganda).

Metrik:
- **Exact match** — nomor plat terbaca persis sama dengan label.
- **CER** (*Character Error Rate*) — jumlah edit karakter (Levenshtein) dibagi jumlah karakter label. Makin kecil makin baik.
- **Precision / Recall / F1 karakter** — karakter tebakan dan label disejajarkan dulu dengan algoritma edit distance, lalu dihitung yang cocok.
- Angka ditampilkan dua versi: **semua plat** (plat yang ditolak "TIDAK VALID" dihitung salah) dan **hanya yang lolos validasi** (cara hitung `evaluate_plate_tax.py`).
- **Angka pajak terbaca** — dataset tidak punya label pajak, jadi yang dihitung hanya yang berhasil terbaca dalam format MM-YY.

In [ ]:
import easyocr
from Script import plate_tax_pipeline as baru

def muat_versi_lama(ref):
    src = subprocess.run(["git", "show", f"{ref}:Script/plate_tax_pipeline.py"], cwd=BASE,
                         capture_output=True, text=True, encoding="utf-8")
    if src.returncode != 0:
        raise RuntimeError(f"Gagal membaca versi lama ({ref}): {src.stderr}")
    mod = types.ModuleType("plate_tax_pipeline_lama")
    exec(compile(src.stdout, "plate_tax_pipeline_lama.py", "exec"), mod.__dict__)
    return mod

lama = muat_versi_lama(BASELINE_REF)
reader = easyocr.Reader(["en"], gpu=(DEVICE == 0), verbose=False)

def iou(a, b):
    x1, y1, x2, y2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0

def bersih(s):
    return re.sub(r"[^A-Z0-9]", "", str(s).upper())

rows4, t0 = [], time.time()
for ip in sorted(glob.glob(os.path.join(DATA_DIR, "images", SPLIT, "*.jpg"))):
    if len(rows4) >= N_PLAT_OCR:
        break
    stem = os.path.splitext(os.path.basename(ip))[0]
    lab = os.path.join(DATA_DIR, "labelswithLP", SPLIT, stem + ".txt")
    if not os.path.exists(lab):
        continue
    img = cv2.imread(ip)
    gts = [(box, bersih(gt)) for box, _, gt in _crops_from_labels(ip, lab)]
    dets = det.detect(img)
    dipakai = set()
    for gbox, gt in gts:
        if len(rows4) >= N_PLAT_OCR:
            break
        cocok = max(((iou(gbox, d["box"]), i) for i, d in enumerate(dets) if i not in dipakai), default=(0, -1))
        row = {"plat": stem, "gt": gt, "terdeteksi": cocok[0] >= 0.5}
        if row["terdeteksi"]:
            dipakai.add(cocok[1])
            d = dets[cocok[1]]
            r_l = lama.analyze_single_plate(d["deskewed"], reader)
            r_b = baru.analyze_single_plate(d["deskewed"], reader, skip_deskew=True)
            row.update({"lama": bersih(r_l["plate_text"]) if r_l["plate_valid"] else "", "lama_valid": r_l["plate_valid"],
                        "lama_pajak": r_l["tax_text"] if r_l["tax_valid"] else "",
                        "baru": bersih(r_b["plate_text"]) if r_b["plate_valid"] else "", "baru_valid": r_b["plate_valid"],
                        "baru_pajak": r_b["tax_text"] if r_b["tax_valid"] else ""})
        else:
            row.update({"lama": "", "lama_valid": False, "lama_pajak": "",
                        "baru": "", "baru_valid": False, "baru_pajak": ""})
        rows4.append(row)
        print(f"[{len(rows4):3d}/{N_PLAT_OCR}] {stem:9s} GT {gt:10s} | lama {row['lama'] or '-':10s} | baru {row['baru'] or '-':10s}"
              f"{'' if row['terdeteksi'] else '  (tidak terdeteksi YOLO)'}")

df4 = pd.DataFrame(rows4)
df4.to_csv(os.path.join(OUT, "4_ocr_per_plat.csv"), index=False)
print(f"\nSelesai {len(df4)} plat dalam {time.time() - t0:.0f} detik")

In [ ]:
def align(gt, pred):
    # penyejajaran Levenshtein: hasilnya pasangan (karakter_label, karakter_tebakan), '-' = tidak ada
    n, m = len(gt), len(pred)
    D = np.zeros((n + 1, m + 1), int); D[:, 0] = range(n + 1); D[0, :] = range(m + 1)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            D[i, j] = min(D[i - 1, j] + 1, D[i, j - 1] + 1, D[i - 1, j - 1] + (gt[i - 1] != pred[j - 1]))
    pairs, i, j = [], n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and D[i, j] == D[i - 1, j - 1] + (gt[i - 1] != pred[j - 1]):
            pairs.append((gt[i - 1], pred[j - 1])); i, j = i - 1, j - 1
        elif i > 0 and D[i, j] == D[i - 1, j] + 1:
            pairs.append((gt[i - 1], "-")); i -= 1
        else:
            pairs.append(("-", pred[j - 1])); j -= 1
    return pairs[::-1], int(D[n, m])

def metrik(df, kol, hanya_valid=False):
    d = df[df[f"{kol}_valid"]] if hanya_valid else df
    if len(d) == 0:
        return {}
    tp = fp = fn = edit = total = 0
    for gt, pr in zip(d["gt"], d[kol]):
        pairs, e = align(gt, pr)
        edit += e; total += max(1, len(gt))
        for g, p in pairs:
            if g == p: tp += 1
            else:
                if p != "-": fp += 1
                if g != "-": fn += 1
    P = tp / (tp + fp) if tp + fp else 0.0
    R = tp / (tp + fn) if tp + fn else 0.0
    return {"jumlah plat": len(d), "exact match": (d["gt"] == d[kol]).mean(), "CER": edit / total,
            "precision karakter": P, "recall karakter": R, "F1 karakter": 2 * P * R / (P + R) if P + R else 0.0,
            "pajak terbaca (MM-YY)": (d[f"{kol}_pajak"] != "").mean()}

tabel = pd.DataFrame({
    "lama (semua plat)": metrik(df4, "lama"), "baru (semua plat)": metrik(df4, "baru"),
    "lama (lolos validasi)": metrik(df4, "lama", True), "baru (lolos validasi)": metrik(df4, "baru", True),
})
tampil = tabel.copy().astype(object)
for idx in tampil.index:
    if idx != "jumlah plat":
        tampil.loc[idx] = [f"{v:.1%}" if pd.notna(v) else "-" for v in tabel.loc[idx]]
    else:
        tampil.loc[idx] = [f"{int(v)}" if pd.notna(v) else "-" for v in tabel.loc[idx]]
print(f"YOLO menemukan {int(df4['terdeteksi'].sum())}/{len(df4)} plat. Plat yang tidak terdeteksi dihitung salah di kolom 'semua plat'.")
display(tampil)
tabel.to_csv(os.path.join(OUT, "4_ringkasan_ocr.csv"))

### Confusion matrix karakter (versi baru)

Baris = karakter sebenarnya, kolom = karakter yang dibaca OCR. Hanya karakter yang **pernah salah baca** yang ditampilkan supaya mudah dibaca.
`-` artinya karakter hilang (baris) atau karakter tambahan (kolom). Pola seperti **6→5**, **D→0**, **N→M** biasanya muncul dari resolusi plat yang kecil.

In [ ]:
from collections import Counter

pasangan = Counter()
for gt, pr in zip(df4["gt"], df4["baru"]):
    if pr:
        for g, p in align(gt, pr)[0]:
            pasangan[(g, p)] += 1

salah = {k: v for k, v in pasangan.items() if k[0] != k[1]}
if not salah:
    print("Tidak ada salah baca karakter pada plat yang terbaca.")
else:
    top = sorted(salah.items(), key=lambda x: -x[1])[:15]
    display(pd.DataFrame([{"sebenarnya": g, "dibaca": p, "jumlah": n} for (g, p), n in top]))
    huruf = sorted({c for (g, p) in salah for c in (g, p)}, key=lambda c: (c == "-", c))
    M = np.array([[pasangan.get((g, p), 0) for p in huruf] for g in huruf])
    fig, ax = plt.subplots(figsize=(0.45 * len(huruf) + 2, 0.45 * len(huruf) + 1.5))
    ax.imshow(M, cmap="Blues")
    ax.set_xticks(range(len(huruf))); ax.set_xticklabels(huruf)
    ax.set_yticks(range(len(huruf))); ax.set_yticklabels(huruf)
    ax.set_xlabel("dibaca OCR"); ax.set_ylabel("sebenarnya")
    for i in range(len(huruf)):
        for j in range(len(huruf)):
            if M[i, j]:
                ax.text(j, i, M[i, j], ha="center", va="center", color="white" if M[i, j] > M.max() / 2 else "black", fontsize=8)
    ax.set_title("Confusion matrix karakter (versi baru)")
    plt.tight_layout(); plt.savefig(os.path.join(OUT, "4_confusion_karakter.png"), dpi=130); plt.show()

## Ringkasan

In [ ]:
ringkas = {"branch": branch, "split": SPLIT, "waktu": datetime.now().isoformat(timespec="seconds"),
           "yolo": yolo_summary,
           "ocr": {k: {m: (float(v) if pd.notna(v) else None) for m, v in tabel[k].items()} for k in tabel.columns}}
with open(os.path.join(OUT, "ringkasan.json"), "w", encoding="utf-8") as f:
    json.dump(ringkas, f, indent=2, ensure_ascii=False)

baris = []
if yolo_summary:
    baris.append(f"- **Deteksi YOLO ({SPLIT})**: Precision {yolo_summary['Precision']:.1%}, Recall {yolo_summary['Recall']:.1%}, "
                 f"F1 {yolo_summary['F1']:.1%}, mAP@0.5 {yolo_summary['mAP@0.5']:.1%}")
b, l = tabel["baru (semua plat)"], tabel["lama (semua plat)"]
baris.append(f"- **OCR nomor plat (semua plat)**: exact match {l['exact match']:.1%} → **{b['exact match']:.1%}**, "
             f"CER {l['CER']:.1%} → **{b['CER']:.1%}**, F1 karakter {l['F1 karakter']:.1%} → **{b['F1 karakter']:.1%}**")
baris.append(f"- **Angka pajak terbaca**: {l['pajak terbaca (MM-YY)']:.1%} → **{b['pajak terbaca (MM-YY)']:.1%}**")
baris.append(f"- Semua file hasil: `{os.path.relpath(OUT, BASE)}`")
display(Markdown("\n".join(baris)))